# Decision Tree Training

Muhammad Sarab Aftab

I am using grid search to tune the decision tree on the training split, then checking the best one on dev. Test is still left alone.


## Loading the data

Same split as the investigation notebooks. I do not scale the data before the search. The scaler stays inside the pipeline so it gets fit on each fold.


In [1]:
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

from dataset_preparation import build_preparation_pipeline, prepare_datasets

data = prepare_datasets()

print("Train rows:", len(data["y_train"]))
print("Dev rows:", len(data["y_dev"]))
print("Test rows, not scored:", len(data["y_test"]))


Train rows: 69138
Dev rows: 23046
Test rows, not scored: 23047


## Grid search

The grid covers both trees we already tried: my unlimited Gini tree, and a shallower entropy tree with balanced class weights. It keeps the settings with the best phishing F1.


In [2]:
model = DecisionTreeClassifier(random_state=42)

param_grid = {
    "model__criterion": ["gini", "entropy"],
    "model__max_depth": [8, 16, None],
    "model__min_samples_split": [2, 50],
    "model__min_samples_leaf": [1, 20],
    "model__max_features": [None, "sqrt"],
    "model__class_weight": [None, "balanced"],
}

full_pipeline = Pipeline(
    steps=[
        ("prepare", build_preparation_pipeline()),
        ("model", model),
    ]
)

search = GridSearchCV(
    full_pipeline,
    param_grid,
    scoring="f1",
    cv=3,
    n_jobs=-1,
)
search.fit(data["X_train"], data["y_train"])

print("Best cross-validation F1:", round(search.best_score_, 4))
print("Best settings:")
for name, value in search.best_params_.items():
    print(f"  {name}: {value}")


Best cross-validation F1: 0.8971
Best settings:
  model__class_weight: None
  model__criterion: entropy
  model__max_depth: 16
  model__max_features: None
  model__min_samples_leaf: 1
  model__min_samples_split: 2


## Dev scores

These are the best tree on the dev split. Phishing is the class I care about.


In [3]:
best_tree = search.best_estimator_
dev_predictions = best_tree.predict(data["X_dev"])
dev_probability = best_tree.predict_proba(data["X_dev"])[:, 1]
y_dev = data["y_dev"]

dev_scores = pd.Series(
    {
        "accuracy": accuracy_score(y_dev, dev_predictions),
        "precision": precision_score(y_dev, dev_predictions, pos_label=1),
        "recall": recall_score(y_dev, dev_predictions, pos_label=1),
        "f1": f1_score(y_dev, dev_predictions, pos_label=1),
        "roc_auc": roc_auc_score(y_dev, dev_probability),
    }
).round(4)

dev_scores


accuracy     0.9723
precision    0.9133
recall       0.8924
f1           0.9027
roc_auc      0.9501
dtype: float64

## Conclusion

The search picked an entropy tree with max depth 16, no class weight, and all of the features. On dev that is about 0.972 accuracy, phishing precision about 0.913, recall about 0.892, F1 about 0.903, and ROC AUC about 0.950.

F1 is about the same as my first tree with no depth limit, and the AUC is a bit higher. This is the tree I would take forward. I did not score the test split.
